<a href="https://colab.research.google.com/github/aqilalhafidz/ssf-uns-2026-tma-forecasting/blob/main/ssf_uns_2026_tma_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Water Level Forecasting for the Bengawan Solo River Basin (SSF UNS 2026)

Forecasting water levels at 30 river gauges (`tma_mdpl`, metres above sea level; metric: RMSE; horizon: 8 months, 3 observations per day).
Column names `nama_pos` (station name) and `tma_mdpl` (water level) follow the competition data.

**Model.** LightGBM with 56 features; target = water level minus the station mean; label spikes removed.
The final prediction is the average of **four models trained with different cut-off dates** (2024-01-18, 2024-09-19, 2025-01-18, full training set), each averaged over **5 seeds** with bagging.

**Leaderboard score of this notebook's output** (submitted after the deadline): public 1.60479, private 1.66363.

Decisions and evidence (three forward-chaining validation folds with a fixed number of trees; not the leaderboard):

| Decision | Evidence |
|---|---|
| Target minus the station mean | Removes the fragility of the raw-target model (fold 1 RMSE 2.85 -> 0.77, fold 2 1.62 -> 1.00) |
| Spike rule: deviation > 30 m from the 5-observation median, or a sudden zero/negative value | Keeps valid recovery rows and leaves small jumps that may be real flood peaks |
| Long-window rainfall sums use full windows only | Partial windows understated early-2023 rainfall; mean 3-fold RMSE 0.814 -> 0.771 after the fix |
| Basin-average rainfall features | Mean 3-fold RMSE 0.850 -> 0.770, consistent in folds 2 and 3 |
| Drop `solar_radiation_mj_m2` | Contains the code -999 from 2025-12-31 (57% of test rows) |
| Drop `nino_34` and MJO features | Better without them in all three folds once the features were cleaned |
| No river-network features from the HydroRIVERS shapefile | Worse than basin-average rainfall (0.813 vs 0.770) |
| Default model parameters | Optuna tuning did not beat the defaults across 3 seeds (0.8419 vs 0.8372) |
| Antecedent precipitation index (API: exponentially weighted rainfall, half-lives 12-1440 h, station and basin level) | Single-model 3-fold RMSE 0.7698 -> 0.7454; folds 2 and 3 improved by 0.034 and 0.037; Wonogiri Dam RMSE 1.54 -> 1.22 |
| Observation hour and astronomical tide harmonics (M2, S2, N2, K1, O1) | Arjowinangun - Pacitan is clearly tidal (Lomb-Scargle power ratio 652 at M2); 3-fold RMSE 0.7454 -> 0.7405 (lower in all folds); rainy season 0.8855 -> 0.8770; Arjowinangun 0.458 -> 0.433 |
| Average over 5 seeds with bagging (subsample 0.8, colsample 0.8) | RMSE 0.7698 -> 0.7624, better in all three folds; the leaderboard MSE dropped by a comparable amount |
| Average over models with different cut-off dates | 2025-01-18 window: 0.7556 (latest model only) -> 0.7104; the model with the least data was the best single model; the public leaderboard score improved by 0.0085 |
| No post-processing (clipping, linear calibration, smoothing) | Clipping +0.002, calibration +0.017 (slope unstable: 1.09 / 0.885 / 0.932), smoothing neutral |
| No CatBoost or linear-model blend | CatBoost 8-12% worse than LightGBM, linear models far worse; no blend beat LightGBM alone |
| Mean (not median) across cut-off models | Median 0.7295 vs mean 0.7104 in the 2025-01-18 window (median helped Kali Anyar, hurt Wonogiri Dam) |
| No per-station blend | A 70/30 blend improved validation RMSE by about 0.009, but its leaderboard score (1.60232 / 1.66202) was essentially equal to the version without it (1.60199 / 1.66234) |
| No wider ensemble of training windows (closer cut-offs, shifted windows) | Worse: 0.8344 / 0.9315 / 0.8157 vs 0.7822 for the baseline set in the 2024-09-19 window (weak members dilute the average) |
| No block bagging (models trained without one quarter) | Did not beat cut-off averaging (0.8242 / 0.7287 vs 0.8185 / 0.7104) |

Some comparisons were made at earlier stages of feature engineering, so absolute values differ between rows; each row compares alternatives within the same stage.

Notes:
- Large validation gains from features barely appeared on the leaderboard, whereas variance reduction (averaging over seeds and cut-offs) did. The cause is not established.
- Averaging the predictions of two equally accurate models lowers the MSE by a quarter of the squared distance between them (checked on two variants of this pipeline: 0.0065 / 0.0057 observed vs 0.0059 predicted), so little variance reduction remains within this model family. An average of two closely related variants (with and without the API features) scored 1.60122 / 1.66091.

Output: `submission_final.csv`.

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # not running in Google Colab

# Configuration
DATA_DIR = '/content/drive/MyDrive/latihan_machine_learning/ssf_uns_2026'   # folder with train.csv, test.csv, sample_submission.csv, data_pendukung/
OUTPUT_DIR = DATA_DIR
REFERENCE_SUBMISSION = None   # optional path to another submission file, compared with the output at the end

SPIKE_LIMIT = 30             # deviation from the 5-observation median treated as a spike (metres)
DIFF_LIMIT = 1.5             # flag a station when a cut-off model differs from the final average by more than this RMSE
SEED = 42
RUN_CUTOFF_CV = True         # False skips the cut-off averaging validation (15 model fits)

# single model for the 3-fold validation: LightGBM defaults, fixed number of trees
PARAM = dict(n_estimators=300, learning_rate=0.05, random_state=SEED, verbose=-1)
# models for the final prediction: average over 5 seeds with bagging
SEED_LIST = [1, 2, 3, 4, 5]
PARAM_BAG = dict(n_estimators=300, learning_rate=0.05, subsample=0.8, subsample_freq=1,
                 colsample_bytree=0.8, verbose=-1)
PRODUCTION_CUTOFFS = ['2024-01-18', '2024-09-19', '2025-01-18', None]   # None = full cleaned training set

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Load data

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
env_data = pd.read_csv(f'{DATA_DIR}/data_pendukung/data_lingkungan.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
env_data['datetime'] = pd.to_datetime(env_data['datetime'])

# the test id has the form 'YYYY-MM-DD HH:MM:SS - Station name'
test[['datetime', 'nama_pos']] = test['id'].str.split(' - ', n=1, expand=True)
test['datetime'] = pd.to_datetime(test['datetime'])

print('train    :', train.shape, train['datetime'].min(), 'to', train['datetime'].max())
print('test     :', test.shape, test['datetime'].min(), 'to', test['datetime'].max())
print('env data :', env_data.shape, env_data['datetime'].min(), 'to', env_data['datetime'].max())

train    : (84396, 3) 2023-01-01 06:00:00 to 2025-09-18 18:00:00
test     : (21780, 3) 2025-09-19 06:00:00 to 2026-05-18 18:00:00
env data : (888480, 27) 2023-01-01 00:00:00 to 2026-05-18 23:00:00


### Raw data checks

In [ ]:
print('NaN in tma_mdpl (train):', train['tma_mdpl'].isna().sum())

print('Duplicate keys, train    :', train.duplicated(['datetime', 'nama_pos']).sum())
print('Duplicate keys, test     :', test.duplicated(['datetime', 'nama_pos']).sum())
print('Duplicate keys, env data :', env_data.duplicated(['datetime', 'nama_pos']).sum())

stations_train = set(train['nama_pos'])
stations_test = set(test['nama_pos'])
stations_env = set(env_data['nama_pos'])
print('Stations in train / test / env:', len(stations_train), len(stations_test), len(stations_env))
print('Test stations missing from train:', stations_test - stations_train)
print('Train stations missing from env :', stations_train - stations_env)

print('Observation hours in train:', sorted(train['datetime'].dt.hour.unique()))
print('Observation hours in test :', sorted(test['datetime'].dt.hour.unique()))
print('Test ids match sample_submission ids:', set(test['id']) == set(sample_sub['id']))

nan_counts = env_data.isna().sum()
print('NaN counts in env data:')
print(nan_counts[nan_counts > 0])

# missing-value codes (for example -999) in numeric columns of the environmental data
sentinel_hits = []
for col in env_data.select_dtypes('number').columns:
    mask = env_data[col] <= -900
    if mask.sum() > 0:
        sentinel_hits.append({'column': col, 'rows': int(mask.sum()), 'percent': round(mask.mean() * 100, 2),
                              'first_seen': env_data.loc[mask, 'datetime'].min()})
if len(sentinel_hits) > 0:
    print('WARNING: columns with values <= -900 (missing-value code?):')
    print(pd.DataFrame(sentinel_hits).to_string())
else:
    print('No values <= -900 in numeric columns of the environmental data')

# the basin-average features use all stations per hour: every hour should have the same number of stations
print('Stations per hour in env data (unique values):', env_data.groupby('datetime').size().unique())

# gaps in the training target (time jumps > 2 days per station)
train_sorted = train.sort_values(['nama_pos', 'datetime'])
time_delta = train_sorted.groupby('nama_pos')['datetime'].diff()
has_gap = time_delta > pd.Timedelta(days=2)
gaps = train_sorted.loc[has_gap, ['nama_pos', 'datetime']].copy()
gaps['gap_days'] = time_delta[has_gap].dt.days
print('Gaps longer than 2 days in train:', len(gaps))
print(gaps.sort_values('gap_days', ascending=False).head(10))

NaN in tma_mdpl (train): 0
Duplicate keys, train    : 0
Duplicate keys, test     : 0
Duplicate keys, env data : 0
Stations in train / test / env: 30 30 30
Test stations missing from train: set()
Train stations missing from env : set()
Observation hours in train: [np.int32(6), np.int32(12), np.int32(18)]
Observation hours in test : [np.int32(6), np.int32(12), np.int32(18)]
Test ids match sample_submission ids: True
NaN counts in env data:
soil_moisture_0_7cm          720
soil_moisture_7_28cm         720
soil_moisture_28_100cm       720
soil_moisture_100_255cm      720
surface_pressure_hpa         720
pressure_msl_hpa             720
rmm1                         720
rmm2                         720
mjo_phase                    720
mjo_amplitude                720
mjo_active                   720
nino_34                    12960
dtype: int64
                  column    rows  percent first_seen
0  solar_radiation_mj_m2  100080    11.26 2025-12-31
Stations per hour in env data (unique value

## 2. Fill gaps and build rolling features

In [ ]:
# Sort by station and time (required before forward-fill and rolling windows)
env_data = env_data.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

hour_gaps = env_data.groupby('nama_pos')['datetime'].diff().dropna()
print('Time differences between consecutive rows:', hour_gaps.value_counts().to_dict())
print('Rows per station (min, max):', env_data.groupby('nama_pos').size().min(), env_data.groupby('nama_pos').size().max())

# forward-fill per station for feature columns with missing values (the last day is empty)
ffill_cols = [
    'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm',
    'surface_pressure_hpa', 'pressure_msl_hpa',
]
nan_before = env_data[ffill_cols].isna().sum()
env_data[ffill_cols] = env_data.groupby('nama_pos')[ffill_cols].ffill()
nan_after = env_data[ffill_cols].isna().sum()
print('NaN before forward-fill (non-zero):')
print(nan_before[nan_before > 0])
print('NaN after forward-fill (non-zero):')
print(nan_after[nan_after > 0])

Time differences between consecutive rows: {Timedelta('0 days 01:00:00'): 888450}
Rows per station (min, max): 29616 29616
NaN before forward-fill (non-zero):
soil_moisture_0_7cm        720
soil_moisture_7_28cm       720
soil_moisture_28_100cm     720
soil_moisture_100_255cm    720
surface_pressure_hpa       720
pressure_msl_hpa           720
dtype: int64
NaN after forward-fill (non-zero):
Series([], dtype: int64)


In [ ]:
# Rolling features per station (windows in hours).
# Rainfall sums use full windows only (NaN until the window is filled); means may use partial windows.
g = env_data.groupby('nama_pos')

env_data['humidity_pct_mean72'] = g['humidity_pct'].transform(lambda x: x.rolling(72).mean())
env_data['soil_moisture_0_7cm_mean36'] = g['soil_moisture_0_7cm'].transform(lambda x: x.rolling(36).mean())
env_data['rainfall_mm_sum48'] = g['rainfall_mm'].transform(lambda x: x.rolling(48).sum())

for name, w in {'rainfall_mm_sum168': 168, 'rainfall_mm_sum336': 336, 'rainfall_mm_sum720': 720,
                'rainfall_mm_sum1440': 1440, 'rainfall_mm_sum2160': 2160}.items():
    env_data[name] = g['rainfall_mm'].transform(lambda x: x.rolling(w).sum())

env_data['soil_moisture_0_7cm_mean168'] = g['soil_moisture_0_7cm'].transform(lambda x: x.rolling(168, min_periods=84).mean())
env_data['soil_moisture_28_100cm_mean720'] = g['soil_moisture_28_100cm'].transform(lambda x: x.rolling(720, min_periods=360).mean())
env_data['soil_moisture_100_255cm_mean720'] = g['soil_moisture_100_255cm'].transform(lambda x: x.rolling(720, min_periods=360).mean())

ROLLING_FEATURES = ['humidity_pct_mean72', 'soil_moisture_0_7cm_mean36', 'rainfall_mm_sum48']
LONG_FEATURES = ['rainfall_mm_sum168', 'rainfall_mm_sum336', 'rainfall_mm_sum720',
                 'rainfall_mm_sum1440', 'rainfall_mm_sum2160',
                 'soil_moisture_0_7cm_mean168', 'soil_moisture_28_100cm_mean720',
                 'soil_moisture_100_255cm_mean720']

# Antecedent precipitation index (API): exponentially weighted mean of rainfall per station (half-life in hours).
# min_periods = half-life: the first half-life is left as NaN so the start of the series is not biased.
HALFLIFE = [12, 48, 168, 720, 1440]
for h in HALFLIFE:
    env_data[f'api_station_h{h}'] = g['rainfall_mm'].transform(lambda x: x.ewm(halflife=h, min_periods=h).mean())
API_STATION_FEATURES = [f'api_station_h{h}' for h in HALFLIFE]

In [ ]:
# Basin-average rainfall: mean rainfall over all stations per hour, then summed over full windows
basin_rain = env_data.groupby('datetime')['rainfall_mm'].mean().sort_index()
basin = pd.DataFrame({'datetime': basin_rain.index})
for name, w in {'basin_rain_sum24': 24, 'basin_rain_sum72': 72, 'basin_rain_sum168': 168,
                'basin_rain_sum720': 720, 'basin_rain_sum2160': 2160}.items():
    basin[name] = basin_rain.rolling(w).sum().to_numpy()
basin['basin_rain_sum24_lag24'] = basin['basin_rain_sum24'].shift(24)   # rainfall 24-48 hours ago
basin['basin_rain_sum24_lag48'] = basin['basin_rain_sum24'].shift(48)   # rainfall 48-72 hours ago
BASIN_FEATURES = [k for k in basin.columns if k != 'datetime']

# Basin-level API: exponentially weighted basin rainfall (same half-lives as the station-level API)
for h in HALFLIFE:
    basin[f'api_basin_h{h}'] = basin_rain.ewm(halflife=h, min_periods=h).mean().to_numpy()
API_BASIN_FEATURES = [f'api_basin_h{h}' for h in HALFLIFE]

print('Hours in basin table:', len(basin), '| unique time differences:', basin['datetime'].diff().dropna().unique())
print(basin[BASIN_FEATURES + API_BASIN_FEATURES].describe().T[['count', 'mean', 'min', 'max']])

Hours in basin table: 29616 | unique time differences: <TimedeltaArray>
['0 days 01:00:00']
Length: 1, dtype: timedelta64[ns]
                          count        mean           min          max
basin_rain_sum24        29593.0    6.666141  0.000000e+00    53.010000
basin_rain_sum72        29545.0   20.007899  0.000000e+00    90.183333
basin_rain_sum168       29449.0   46.709960  0.000000e+00   146.890000
basin_rain_sum720       28897.0  201.164141  1.213333e+00   515.093333
basin_rain_sum2160      27457.0  589.421637  1.561000e+01  1234.130000
basin_rain_sum24_lag24  29569.0    6.670172  0.000000e+00    53.010000
basin_rain_sum24_lag48  29545.0    6.672838  0.000000e+00    53.010000
api_basin_h12           29605.0    0.277826  7.292336e-08     2.197870
api_basin_h48           29569.0    0.278079  4.344478e-04     1.132047
api_basin_h168          29449.0    0.278851  1.993728e-03     0.781228
api_basin_h720          28897.0    0.280443  2.359673e-02     0.579675
api_basin_h1440       

## 3. Merge into train and test

In [ ]:
BASE_FEATURES = [
    'rainfall_mm', 'humidity_pct', 'wind_direction_deg', 'dew_point_c', 'cloud_cover_pct',
    'temperature_c', 'wind_speed_kmh', 'rainfall_max_24h_mm',
    'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm',
    'surface_pressure_hpa', 'pressure_msl_hpa',
]
# Observation hour and astronomical tide harmonics (periods in hours). Deterministic, so valid for the test period.
TIDE_PERIODS = {'M2': 12.4206, 'S2': 12.0, 'N2': 12.6583, 'K1': 23.9345, 'O1': 25.8193}
T0 = pd.Timestamp('2023-01-01')
TIDE_FEATURES = ['obs_hour'] + [f'{fn}_{k}' for k in TIDE_PERIODS for fn in ('sin', 'cos')]

FEATURES = (['nama_pos'] + BASE_FEATURES + ['month', 'dayofyear'] + ROLLING_FEATURES + LONG_FEATURES
            + BASIN_FEATURES + API_STATION_FEATURES + API_BASIN_FEATURES + TIDE_FEATURES)
print('Number of features:', len(FEATURES))

env_features = env_data[['datetime', 'nama_pos'] + BASE_FEATURES + ROLLING_FEATURES + LONG_FEATURES + API_STATION_FEATURES]

n_train, n_test = len(train), len(test)
train = train.merge(env_features, on=['datetime', 'nama_pos'], how='left').merge(basin, on='datetime', how='left')
test = test.merge(env_features, on=['datetime', 'nama_pos'], how='left').merge(basin, on='datetime', how='left')
print('Train rows before/after merge:', n_train, len(train))
print('Test rows before/after merge :', n_test, len(test))

for df in (train, test):
    df['month'] = df['datetime'].dt.month
    df['dayofyear'] = df['datetime'].dt.dayofyear
    hours = (df['datetime'] - T0).dt.total_seconds().to_numpy() / 3600.0
    df['obs_hour'] = df['datetime'].dt.hour
    for k, period in TIDE_PERIODS.items():
        df[f'sin_{k}'] = np.sin(2 * np.pi * hours / period)
        df[f'cos_{k}'] = np.cos(2 * np.pi * hours / period)

# station as a categorical feature with the same categories in train and test
station_dtype = pd.CategoricalDtype(sorted(train['nama_pos'].unique()))
train['nama_pos'] = train['nama_pos'].astype(station_dtype)
test['nama_pos'] = test['nama_pos'].astype(station_dtype)
print('Test rows with an unknown station:', test['nama_pos'].isna().sum())

Number of features: 56
Train rows before/after merge: 84396 84396
Test rows before/after merge : 21780 21780
Test rows with an unknown station: 0


In [ ]:
numeric_features = [f for f in FEATURES if f != 'nama_pos']
for name, df in [('train', train), ('test', test)]:
    n = df[FEATURES].isna().sum()
    print(f'NaN counts in {name} features (non-zero):')
    print(n[n > 0])
    sentinel = (df[numeric_features] <= -900).sum()
    print(f'Features with values <= -900 in {name}:', sentinel[sentinel > 0].to_dict() if (sentinel > 0).any() else 'none')

# share of test rows with feature values outside the training range (5 largest)
lower, upper = train[numeric_features].min(), train[numeric_features].max()
outside = ((test[numeric_features] < lower) | (test[numeric_features] > upper)).mean() * 100
print('Percent of test rows outside the training range (5 largest):')
print(outside.sort_values(ascending=False).head(5).round(2))

NaN counts in train features (non-zero):
humidity_pct_mean72                 261
soil_moisture_0_7cm_mean36          116
rainfall_mm_sum48                   174
rainfall_mm_sum168                  609
rainfall_mm_sum336                 1218
rainfall_mm_sum720                 2610
rainfall_mm_sum1440                5194
rainfall_mm_sum2160                7781
soil_moisture_0_7cm_mean168         290
soil_moisture_28_100cm_mean720     1305
soil_moisture_100_255cm_mean720    1305
basin_rain_sum24                     87
basin_rain_sum72                    261
basin_rain_sum168                   609
basin_rain_sum720                  2610
basin_rain_sum2160                 7781
basin_rain_sum24_lag24              174
basin_rain_sum24_lag48              261
api_station_h12                      29
api_station_h48                     174
api_station_h168                    609
api_station_h720                   2610
api_station_h1440                  5194
api_basin_h12                        29

## 4. Remove label spikes

A row is treated as a spike when its water level deviates by more than `SPIKE_LIMIT` from the median of the 5 surrounding observations (per station),
or when it is a sudden zero or negative value. Small jumps (+3 to +5 m) are kept on purpose because they can be real flood peaks.

In [ ]:
train = train.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

g_target = train.groupby('nama_pos', observed=True)['tma_mdpl']
median5 = g_target.transform(lambda x: x.rolling(5, center=True, min_periods=3).median())

is_spike = ((train['tma_mdpl'] - median5).abs() > SPIKE_LIMIT) | ((train['tma_mdpl'] <= 0) & (median5 > 1))

print('Spike rows:', is_spike.sum(), 'of', len(train), f'({is_spike.mean() * 100:.3f}%)')
spike_rows = train.loc[is_spike, ['datetime', 'nama_pos', 'tma_mdpl']].copy()
spike_rows['median5'] = median5[is_spike]
print(spike_rows)

train_clean = train[~is_spike].copy()
print('Rows in train_clean:', len(train_clean))

Spike rows: 22 of 84396 (0.026%)
                 datetime                   nama_pos    tma_mdpl    median5
15016 2023-09-14 06:00:00   Bojonegoro - Kali Kethek    0.000000   6.790000
15226 2023-11-23 12:00:00   Bojonegoro - Kali Kethek  225.130000   6.550000
15240 2023-11-28 06:00:00   Bojonegoro - Kali Kethek  174.130000   7.170000
27921 2025-06-15 18:00:00          Floodway Bridge C   47.000000   4.431908
29491 2023-02-22 18:00:00                      Jarum  250.140000  91.140000
29539 2023-03-10 18:00:00                      Jarum  150.140000  90.890000
29541 2023-03-11 12:00:00                      Jarum  135.140000  90.542677
29725 2023-05-11 18:00:00                      Jarum  200.395917  91.140000
34492 2025-02-28 06:00:00                      Jurug    0.000000  79.650000
35740 2023-08-02 12:00:00                   Kajangan  172.770000  50.133060
38492 2023-06-13 18:00:00  Kali Anyar - Kreteg Abang  323.207409  87.640000
41331 2023-05-24 06:00:00           Kali Pepe - PTPN  1

## 5. Validation: three forward folds

Each fold trains on data before the cut-off and validates on the 8 months after it (the same gap as between training and test).
Target = water level minus the station mean (computed from the training part of the fold only). The number of trees is fixed.
Fold 2 is the closest to the test period (a rainy season right after the training data ends). Stations without data in a fold's training part are not scored.

In [ ]:
FOLDS = [
    ('2024-01-18', '2024-09-18'),
    ('2024-09-19', '2025-05-18'),   # closest to the test period
    ('2025-01-18', '2025-09-18'),
]
RAINY_MONTHS = [11, 12, 1, 2, 3, 4]
FOCUS_STATIONS = ['Wonogiri Dam', 'Bojonegoro - Kali Kethek']

cv_rows = []
for cutoff, end in FOLDS:
    end_exclusive = pd.Timestamp(end) + pd.Timedelta(days=1)
    tr = train[(train['datetime'] < cutoff) & ~is_spike]
    va = train[(train['datetime'] >= cutoff) & (train['datetime'] < end_exclusive) & ~is_spike]

    n_before = len(va)
    va = va[va['nama_pos'].isin(tr['nama_pos'].unique())]
    if len(va) < n_before:
        print(f'[fold {cutoff}] {n_before - len(va)} validation rows dropped (station not in the fold training data)')

    station_mean = tr.groupby('nama_pos', observed=True)['tma_mdpl'].mean()
    offset_tr = tr['nama_pos'].map(station_mean).astype(float).to_numpy()
    offset_va = va['nama_pos'].map(station_mean).astype(float).to_numpy()

    model = lgb.LGBMRegressor(**PARAM)
    model.fit(tr[FEATURES], tr['tma_mdpl'] - offset_tr)
    pred = model.predict(va[FEATURES]) + offset_va

    sq = (va['tma_mdpl'].to_numpy() - pred) ** 2
    rainy = va['datetime'].dt.month.isin(RAINY_MONTHS).to_numpy()
    row = {'fold': cutoff, 'rmse': sq.mean() ** 0.5, 'rmse_rainy': sq[rainy].mean() ** 0.5}
    for station in FOCUS_STATIONS:
        in_station = (va['nama_pos'] == station).to_numpy()
        row[station] = sq[in_station].mean() ** 0.5 if in_station.any() else np.nan
    cv_rows.append(row)

    if cutoff == '2024-09-19':
        va_fold2 = va[['nama_pos', 'datetime', 'tma_mdpl']].copy()
        va_fold2['sq'] = sq

cv_table = pd.DataFrame(cv_rows).set_index('fold')
cv_table.loc['mean'] = cv_table.mean()
print(cv_table.round(3))

[fold 2024-01-18] 106 validation rows dropped (station not in the fold training data)
             rmse  rmse_rainy  Wonogiri Dam  Bojonegoro - Kali Kethek
fold                                                                 
2024-01-18  0.702       0.833         1.374                     1.118
2024-09-19  0.809       0.873         1.188                     1.295
2025-01-18  0.711       0.925         1.122                     1.203
mean        0.741       0.877         1.228                     1.205


In [ ]:
# Error by station in fold 2 (10 largest RMSE)
station_fold2 = va_fold2.groupby('nama_pos', observed=True)['sq'].agg(['size', 'sum', 'mean'])
station_fold2['rmse'] = station_fold2['mean'] ** 0.5
station_fold2['sse_share_pct'] = station_fold2['sum'] / station_fold2['sum'].sum() * 100
print(station_fold2[['size', 'rmse', 'sse_share_pct']].sort_values('rmse', ascending=False).head(10).round(3))

                          size   rmse  sse_share_pct
nama_pos                                            
Bojonegoro - Kali Kethek   638  1.295          8.385
Floodway Bridge C          651  1.260          8.107
Wonogiri Dam               651  1.188          7.204
Ketonggo                   651  1.096          6.126
Napel                      651  1.049          5.614
Gunungsari                 651  1.044          5.565
Kedungupit                 651  1.039          5.513
Sumberrejo                 639  1.020          5.207
Jurug                      652  0.989          5.005
Karangnongko               651  0.932          4.433


## 6. Cut-off averaging validation (optional)

Three models (cut-offs 2024-01-18, 2024-09-19, 2025-01-18), each averaged over 5 seeds, are evaluated on the fold 2 and fold 3 validation windows,
individually and averaged. A model is used only on windows that start after its own training data ends, so there is no leakage.
Rows from stations that are missing from one of the models (Gunungsari) are not scored, to keep the comparison fair.
Set `RUN_CUTOFF_CV = False` to skip this section.

In [ ]:
if RUN_CUTOFF_CV:
    cutoff_names = {'M1': '2024-01-18', 'M2': '2024-09-19', 'M3': '2025-01-18'}

    cutoff_models = {}
    for k, c in cutoff_names.items():
        tr = train[(train['datetime'] < c) & ~is_spike]
        station_mean = tr.groupby('nama_pos', observed=True)['tma_mdpl'].mean()
        offset_tr = tr['nama_pos'].map(station_mean).astype(float).to_numpy()
        fitted = []
        for s in SEED_LIST:
            m = lgb.LGBMRegressor(random_state=s, **PARAM_BAG)
            m.fit(tr[FEATURES], tr['tma_mdpl'] - offset_tr)
            fitted.append(m)
        cutoff_models[k] = {'models': fitted, 'station_mean': station_mean}
        print(f'{k}: model with cut-off {c} fitted ({len(tr)} training rows)')

    def predict_cutoff(k, df):
        info = cutoff_models[k]
        offset = df['nama_pos'].map(info['station_mean']).astype(float).to_numpy()   # NaN for stations absent from the training data
        z = np.mean([m.predict(df[FEATURES]) for m in info['models']], axis=0)
        return offset + z

    COMBOS = [('M1',), ('M2',), ('M3',), ('M1', 'M2'), ('M2', 'M3'), ('M1', 'M3'), ('M1', 'M2', 'M3')]
    results_all, results_rainy = {}, {}
    for window_start, window_end in FOLDS[1:]:          # fold 2 and fold 3 windows
        end_exclusive = pd.Timestamp(window_end) + pd.Timedelta(days=1)
        va = train[(train['datetime'] >= window_start) & (train['datetime'] < end_exclusive) & ~is_spike]
        allowed = [k for k, c in cutoff_names.items() if pd.Timestamp(c) <= pd.Timestamp(window_start)]
        pred = {k: predict_cutoff(k, va) for k in allowed}
        available = np.all([~np.isnan(pred[k]) for k in allowed], axis=0)
        print(f'[window {window_start}] models allowed: {allowed}; rows dropped because a station is missing from a model: {(~available).sum()}')

        y = va['tma_mdpl'].to_numpy()[available]
        rainy = va['datetime'].dt.month.isin(RAINY_MONTHS).to_numpy()[available]
        for combo in COMBOS:
            if not all(k in allowed for k in combo):
                continue
            p = np.mean([pred[k][available] for k in combo], axis=0)
            sq = (y - p) ** 2
            name = '+'.join(combo)
            results_all.setdefault(name, {})[window_start] = sq.mean() ** 0.5
            results_rainy.setdefault(name, {})[window_start] = sq[rainy].mean() ** 0.5

    order = [('+'.join(k)) for k in COMBOS if '+'.join(k) in results_all]
    print('Cut-offs:', cutoff_names)
    print('RMSE, all months (columns = validation window start):')
    print(pd.DataFrame(results_all).T.loc[order].round(4))
    print('RMSE, rainy season:')
    print(pd.DataFrame(results_rainy).T.loc[order].round(4))
else:
    print('Cut-off averaging validation skipped (RUN_CUTOFF_CV = False)')

M1: model with cut-off 2024-01-18 fitted (32442 training rows)
M2: model with cut-off 2024-09-19 fitted (53819 training rows)
M3: model with cut-off 2025-01-18 fitted (64708 training rows)
[window 2024-09-19] models allowed: ['M1', 'M2']; rows dropped because a station is missing from a model: 651
[window 2025-01-18] models allowed: ['M1', 'M2', 'M3']; rows dropped because a station is missing from a model: 657
Cut-offs: {'M1': '2024-01-18', 'M2': '2024-09-19', 'M3': '2025-01-18'}
RMSE, all months (columns = validation window start):
          2024-09-19  2025-01-18
M1            0.8166      0.7249
M2            0.7976      0.7018
M3               NaN      0.7188
M1+M2         0.7848      0.6858
M2+M3            NaN      0.7005
M1+M3            NaN      0.6904
M1+M2+M3         NaN      0.6844
RMSE, rainy season:
          2024-09-19  2025-01-18
M1            0.8794      0.8965
M2            0.8624      0.8790
M3               NaN      0.9409
M1+M2         0.8482      0.8657
M2+M3      

## 7. Final prediction

Four models with different cut-off dates (including the full cleaned training set), each averaged over 5 seeds, then averaged together.
Stations that are absent from the training data of the earliest cut-off model (Gunungsari) use the average of the models that include them.

In [ ]:
pred_cutoff = {}
for c in PRODUCTION_CUTOFFS:
    tr = train_clean if c is None else train_clean[train_clean['datetime'] < c]
    station_mean = tr.groupby('nama_pos', observed=True)['tma_mdpl'].mean()
    offset_tr = tr['nama_pos'].map(station_mean).astype(float).to_numpy()
    offset_test = test['nama_pos'].map(station_mean).astype(float).to_numpy()   # NaN for stations absent from the training data

    z = []
    for s in SEED_LIST:
        m = lgb.LGBMRegressor(random_state=s, **PARAM_BAG)
        m.fit(tr[FEATURES], tr['tma_mdpl'] - offset_tr)
        z.append(m.predict(test[FEATURES]))
    name = 'full' if c is None else c
    pred_cutoff[name] = offset_test + np.mean(z, axis=0)
    print(f'cut-off {name}: training rows = {len(tr)}, test rows without a prediction (station absent from training data) = {np.isnan(pred_cutoff[name]).sum()}')

# average of the four models (nanmean for stations absent from the early cut-off model)
pred_final = np.nanmean(np.vstack([pred_cutoff[k] for k in pred_cutoff]), axis=0)

sub_final = sample_sub[['id']].merge(pd.DataFrame({'id': test['id'], 'tma_mdpl': pred_final}), on='id', how='left')
sub_final.to_csv(f'{OUTPUT_DIR}/submission_final.csv', index=False)
print('Saved: submission_final.csv', sub_final.shape)

cut-off 2024-01-18: training rows = 32442, test rows without a prediction (station absent from training data) = 726
cut-off 2024-09-19: training rows = 53819, test rows without a prediction (station absent from training data) = 0
cut-off 2025-01-18: training rows = 64708, test rows without a prediction (station absent from training data) = 0
cut-off full: training rows = 84374, test rows without a prediction (station absent from training data) = 0
Saved: submission_final.csv (21780, 2)


## 8. Submission checks

In [ ]:
# Difference between each cut-off model and the final average, per station (distance between predictions, not error against the true values)
diff = pd.DataFrame({'nama_pos': test['nama_pos'].to_numpy()})
for k, p in pred_cutoff.items():
    diff['sq_' + k] = np.where(np.isnan(p), np.nan, (p - pred_final) ** 2)
diff_table = diff.groupby('nama_pos', observed=True).mean() ** 0.5
diff_table.columns = [k.replace('sq_', 'rmse_vs_final_') for k in diff_table.columns]
print(diff_table.max(axis=1).sort_values(ascending=False).head(8).round(3).to_frame('largest_difference'))

flagged_stations = diff_table.max(axis=1)[diff_table.max(axis=1) > DIFF_LIMIT].index.tolist()
if len(flagged_stations) > 0:
    print('WARNING: stations whose difference from the final average exceeds', DIFF_LIMIT, '->', flagged_stations)
else:
    print('No station differs from the final average by more than', DIFF_LIMIT)

# format and plausibility checks
train_range = train_clean.groupby('nama_pos', observed=True)['tma_mdpl'].agg(['min', 'max'])
print('Rows                   :', len(sub_final), '(sample:', len(sample_sub), ')')
print('NaN in tma_mdpl        :', sub_final['tma_mdpl'].isna().sum())
print('Ids match sample       :', (sub_final['id'] == sample_sub['id']).all())
print(sub_final['tma_mdpl'].describe())

check = sub_final.copy()
check['nama_pos'] = check['id'].str.split(' - ', n=1).str[1]
check = check.join(train_range, on='nama_pos')
outside = check[(check['tma_mdpl'] < check['min']) | (check['tma_mdpl'] > check['max'])]
print('Predictions outside the training range per station:', len(outside))
if len(outside) > 0:
    print(outside.groupby('nama_pos').size().sort_values(ascending=False).head(10))

                           largest_difference
nama_pos                                     
Gunungsari                              0.567
Kali Anyar - Kreteg Abang               0.557
Wonogiri Dam                            0.528
Kedungupit                              0.479
Jurug                                   0.382
Floodway Bridge C                       0.344
Bojonegoro - Kali Kethek                0.284
Napel                                   0.278
No station differs from the final average by more than 1.5
Rows                   : 21780 (sample: 21780 )
NaN in tma_mdpl        : 0
Ids match sample       : True
count    21780.000000
mean        55.298288
std         46.707035
min          0.762046
25%         10.027328
50%         45.586077
75%         90.675341
max        144.458495
Name: tma_mdpl, dtype: float64
Predictions outside the training range per station: 0


In [ ]:
# Optional comparison with another submission file (set REFERENCE_SUBMISSION in the configuration cell).
if REFERENCE_SUBMISSION is not None and os.path.exists(REFERENCE_SUBMISSION):
    reference = pd.read_csv(REFERENCE_SUBMISSION)
    merged = sub_final.merge(reference, on='id', suffixes=('_new', '_ref'))
    print(f'Comparison with {os.path.basename(REFERENCE_SUBMISSION)}: matching rows = {len(merged)}',
          '| RMSE between predictions =', round(root_mean_squared_error(merged['tma_mdpl_ref'], merged['tma_mdpl_new']), 5),
          '| maximum absolute difference =', round(float((merged['tma_mdpl_ref'] - merged['tma_mdpl_new']).abs().max()), 5))
else:
    print('No reference submission set; comparison skipped.')

No reference submission set; comparison skipped.
